# SIPTA — Validación: Finanzas, Inversión Pública y Comercio Informal (RIVI)
**Proyecto**: Sistema de Indicadores y Priorización Territorial y Alertas Tempranas (DataJam Bogotá)  
**Fase PDCO**: CONTROL | **Fase CRISP-DM**: Data Quality & Validation  
**Autoría**: Persona A (Adan Sánchez — Lead Data Engineer)  
**Fuente**: `data/raw/FINANZAS_INVERSION_PUBLICA/rivi-numero-vendedores-informales-localidad-*.txt`, `inversion_educacion_por_localidad_12_2025.gpkg`  
**Temporalidad**: **Series Semestrales 2017 - 2019 (IPES RIVI) y Corte 12.2025 (Inversión SED)**  
**Indicadores Habilitados**: `FIN-001` (Inversión per cápita), `FIN-002` (Vendedores Informales RIVI por 10k hab)


## 1. Validación de Calidad Técnica con `src/validation/validate_data.py`


In [ ]:
import sys
from pathlib import Path

# Resolver la raíz del proyecto SIPTA
for p in [Path('.').resolve(), Path('.').resolve().parent, Path('.').resolve().parent.parent]:
    if (p / 'src').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import pandas as pd
from src.validation.validate_data import validate_finanzas, inspect_schema, load_raw_finanzas

report = validate_finanzas()
print("=== REPORTE EJECUTIVO DE VALIDACIÓN ===")
print(f"Dominio: {report['domain']}")
print(f"Temporalidad: {report['temporalidad']}")
print(f"Total Registros RIVI: {report['total_rows']}")
print(f"Estado de Calidad: {report['validation_status']}")



## 2. Inspección y Consolidación de Series RIVI


In [ ]:
df_rivi = load_raw_finanzas()
schema_df = inspect_schema(df_rivi)
display(schema_df)



## 3. Demostración de Cálculo de Indicadores (`FIN-001` y `FIN-002`)


In [ ]:
# Agrupación de vendedores informales por Localidad
col_loc = [c for c in df_rivi.columns if 'nombrelocalidad' in c.lower() or 'localidad' in c.lower()][0]
col_vend = [c for c in df_rivi.columns if 'numero' == c.lower() or 'vendedor' in c.lower()][0]

df_rivi[col_vend] = pd.to_numeric(df_rivi[col_vend], errors='coerce').fillna(0)
resumen_rivi = df_rivi.groupby(col_loc)[col_vend].mean().reset_index(name='PROMEDIO_VENDEDORES_RIVI')

display(resumen_rivi.head(10))



## 4. Dictamen de Validez
- **Fuente Válida**: Sí. 6 semestres continuos de caracterización de vendedores informales IPES.
- **Factibilidad de Indicador**: `FIN-001` y `FIN-002` habilitados para medir vulnerabilidad y equidad presupuestal.
